<a href="https://colab.research.google.com/github/notoctane/project/blob/main/decorators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Декораторы в python**

**Декоратор** - это паттерн программирования, который позволяет добавлять новый функционал к нашей функции не видоизменяя саму функцию.

## **Примеры полезных декораторов:**
- Подсчет времени
- Логирование параметров функции
- Повтор функции в случае ошибки
- Ограничение на частый вызов функции

## **Простая напоминалка что такое `*args` и `**kwargs`**

### `*args`

`*args` — много позиционных аргументов

`args` — это просто имя (можно назвать иначе). **Главное — одна звёздочка * перед именем.**

`*args` собирает все переданные позиционные аргументы (те, что без имени) в **кортеж**.

### **Пример**

In [ ]:
def sum_all(*args):
  return sum(args)

print(f'{sum_all(5, 5, 10) = }')

# также можно передавать листы и тд заранее их распокавав

print(f'{sum_all(*[5,5,10]) = }')

sum_all(5, 5, 10) = 20
sum_all(*[5,5,10]) = 20


### **`**kwargs`**

`**kwargs` — много именованных аргументов

`kwargs` — сокращение от *keyword arguments*. Здесь две звёздочки **.

`**kwargs` **собирает** все переданные именованные аргументы (ключ=значение) в **словарь**.

In [ ]:
def dict_items(**kwargs):
  print(f'{kwargs.keys()} : {kwargs.values()}')

dict_items(price=122, color='blue')

dict_keys(['price', 'color']) : dict_values([122, 'blue'])


## **Примеры**

### **Вызов декоратора**

In [ ]:
 # вызвать декоратор можно 2 способами:
@empty_deco
def my_func():
  pass
# Фактически это одно и тоже
my_func = deco(my_func)

### **Простой декоратор**

#### **Скелет декоратора**

In [ ]:
from typing import Callable

# Скелет декоратора
def empty_deco(func: Callable):
  def wrapper():
    res = func()
    return res

  return wrapper

@empty_deco
def my_func():
  return 321

my_func()

322

### **Декоратор с функцией с параметром**

In [ ]:
import time

def deco(func: Callable):
  def wrapper(*args, **kwargs):
    res = func(*args, **kwargs)
    return res

  return wrapper

@deco
def sums(*args):
  return sum(args)

sums(1,2,3,4)

10

In [ ]:
# как это выглядит если мы не используем @

def sums(*args):
  return sum(args)

deco(sums)(1,2,3,4)

10

### **Декоратор с параметром**

In [ ]:
def limit_calls(calls:int):
  ''' Докстринг декоратора'''
  def wrapper(func: Callable):
    def inner(*args, **kwargs):
      '''Докстринг внутренней функции'''
      nonlocal calls # Поскольку он не будет видеть эту переменную
      if calls == 0:
        return

      res = func(*args, **kwargs)
      calls -= 1
      return res
    return inner
  return wrapper

@limit_calls(2)
def sums(*args):
  '''ОЧЕНЬ ВАЖНО'''
  return sum(args)

print(sums(1,2,3,4))

10


#### **Проблема которая возникает**

Если нам нужно будет не потерять очень важный для нас докстринг и название функций, ведь если мы захотим посмотреть докстринг функции с нашим декоратором и какая функция вызывается, то мы увидим:

In [ ]:
sums.__doc__ # Мы здесь видим

'Докстринг внутренней функции'

In [ ]:
sums.__name__ # И название функции также не совпадает

'inner'

Решение воспользоваться ещё одним декоратором:

In [ ]:
from functools import wraps

In [ ]:
def limit_calls(calls:int):
  ''' Докстринг декоратора'''
  def wrapper(func: Callable):
    @wraps(func)
    def inner(*args, **kwargs):
      '''Докстринг внутренней функции'''
      nonlocal calls # Поскольку он не будет видеть эту переменную
      if calls == 0:
        return

      res = func(*args, **kwargs)
      calls -= 1
      return res
    return inner
  return wrapper

@limit_calls(2)
def sums(*args):
  '''Докстринг основной функции'''
  return sum(args)

print(sums(1,2,3,4))

10


In [ ]:
sums.__doc__ # Мы здесь видим

'Докстринг основной функции'

In [ ]:
sums.__name__

'sums'

### **Декоратор для ассинхронных функций**

In [ ]:
import asyncio
from typing import Coroutine

def deco(coroutine: Coroutine):
  async def wraper(*args, **kwargs):
    res = await coroutine(*args, **kwargs)
    return res
  return wraper

@deco
async def my_async_func(time_sleep: int):
  await asyncio.sleep(time_sleep)
  return 1

await my_async_func(2)

1

# **Полезные декораторы**

- `lru_cache` - кэширование данных (`Кэширование` — это временное сохранение результатов «тяжёлых» вычислений или запросов, чтобы при повторном обращении взять их мгновенно, а не пересчитывать заново.)

Кэширование — это специальный приём:

    - мы запоминаем результат для конкретных входных данных (например, аргументов функции),

    - и при повторном вызове с теми же данными достаём из кэша, не пересчитывая.

    - Если ты заранее знаешь, что результат нужен только для одного конкретного набора данных (например, один раз вычислил число π и используешь везде) — сохрани в обычную переменную. Кэш избыточен.

- `contextmanager` или `asynccontextmanager`. Позволяет писать свои контексные менеджеры

In [ ]:
from functools import lru_cache
from contextlib import contextmanager, asynccontextmanager

# **Источник**

https://www.youtube.com/watch?v=VnuDMPQSMjs&list=LL